# PerceptionMetrics basic tutorial
PerceptionMetrics provides a unified evaluation pipeline for perception models. For this basic tutorial, we are going to evaluate a LiDAR segmentation model on the RELLIS-3D test dataset.

## Download required data

#### Dataset (RELLIS-3D)
📝 [Paper](https://arxiv.org/abs/2011.12954) 
 🧑‍💻️ [Repo](https://github.com/unmannedlab/RELLIS-3D)

In [1]:
!pip install gdown

In [ ]:
!mkdir -p local/data && cd local/data

# Download RELLIS-3D LiDAR data

!gdown 1raQJPySyqDaHpc53KPnJVl3Bln6HlcVS -O local/data/  # LiDAR split files
!gdown 1K8Zf0ju_xI5lnx3NTDLJpVTs59wmGPI6 -O local/data/  # ontology
!gdown 12bsblHXtob60KrjV7lGXUQTdC5PhV8Er -O local/data/  # LiDAR labels
!gdown 1lDSVRf_kZrD0zHHMsKJ0V1GN9QATR4wH -O local/data/  # LiDAR points

In [3]:
# Unzip the downloaded archives
!unzip -o local/data/Rellis_3D_lidar_split.zip                                         -d local/data/rellis3d/
!unzip -o local/data/Rellis_3D_ontology.zip                                             -d local/data/rellis3d/
!unzip -o local/data/Rellis_3D_os1_cloud_node_semantickitti_label_id_20210614.zip       -d local/data/rellis3d/
!unzip -o local/data/Rellis_3D_os1_cloud_node_kitti_bin.zip                             -d local/data/rellis3d/

#### Model (PyTorch)

In [4]:
# Download pretrained model
# !gdown <WAITING_FOR_ID> -O local/data/lidar_segmentation_model.pth            # model
# !gdown <WAITING_FOR_ID> -O local/data/lidar_segmentation_model_cfg.json       # configuration
# !gdown <WAITING_FOR_ID> -O local/data/lidar_segmentation_model_ontology.json  # ontology

## Init dataset and model objects

In [ ]:
from perceptionmetrics.datasets import Rellis3DLiDARSegmentationDataset
from perceptionmetrics.models import TorchLiDARSegmentationModel

dataset = Rellis3DLiDARSegmentationDataset(
    dataset_dir="local/data/rellis3d/Rellis-3D",
    split_dir="local/data/rellis3d",
    ontology_fname="local/data/rellis3d/Rellis_3D_ontology/ontology.yaml",
)

model = TorchLiDARSegmentationModel(
    model="local/data/lidar_segmentation_model.pth",
    model_cfg="local/data/lidar_segmentation_model_cfg.json",
    ontology_fname="local/data/lidar_segmentation_model_ontology.json",
)

## Inference

In [ ]:
from matplotlib import pyplot as plt
import numpy as np

from perceptionmetrics.utils import lidar as ul

sample = dataset.dataset.iloc[0]
points_fname = sample['points']
label_fname = sample['label']

points = dataset.read_points(points_fname)
label = dataset.read_label(label_fname)
pred = model.predict(points_fname)

points_xyz = ul.recenter(points[:, :3].copy(), dims=[0, 1])

intensity = points[:, 3] if points.shape[1] > 3 else np.ones(points.shape[0], dtype=np.float32)
intensity = intensity.astype(np.float32)
intensity = (intensity - intensity.min()) / (intensity.max() - intensity.min() + 1e-6)
intensity_rgb = np.stack([intensity, intensity, intensity], axis=1)

dataset_lut = np.array([dataset.ontology[k]['rgb'] for k in sorted(dataset.ontology, key=lambda x: dataset.ontology[x]['idx'])], dtype=np.float32) / 255.0
model_lut = np.array([model.ontology[k]['rgb'] for k in sorted(model.ontology, key=lambda x: model.ontology[x]['idx'])], dtype=np.float32) / 255.0

gt_rgb = dataset_lut[label]
pred_rgb = model_lut[pred]

img_input = ul.render_point_cloud(points_xyz, intensity_rgb.copy(), camera_view='3rd_person', resolution=(960, 540))
img_pred = ul.render_point_cloud(points_xyz, pred_rgb.copy(), camera_view='3rd_person', resolution=(960, 540))
img_gt = ul.render_point_cloud(points_xyz, gt_rgb.copy(), camera_view='3rd_person', resolution=(960, 540))

plt.figure(figsize=(15, 6))
plt.subplot(131), plt.title('Point cloud (intensity)'), plt.imshow(np.array(img_input)), plt.axis('off')
plt.subplot(132), plt.title('Prediction'), plt.imshow(np.array(img_pred)), plt.axis('off')
plt.subplot(133), plt.title('GT'), plt.imshow(np.array(img_gt)), plt.axis('off')
plt.show()

## Evaluation

In [ ]:
results = model.eval(dataset, split="test")
display(results)